# Lab 51: The Skin's Resolution

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-51.ipynb)

**What you will do:** measure your two-point discrimination threshold at several body sites, compare it with the published values in Figure 6.2, and run a toy event-camera simulation that mirrors the body's uneven sensor density.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 51 you used a pair of compasses to find your two-point discrimination threshold -- the
smallest gap at which two simultaneous touches feel like two, rather than one -- on several body
parts. Fingertips need only a couple of millimetres of separation, while the palm, forearm and
back need ten to forty times that gap, because the density of touch receptors and the cortical
territory devoted to each body part vary enormously (Figure 6.2, after Weinstein, 1968).

## Record your results

Test at least the four body parts in Figure 6.2 (fingertip, palm, forearm, back), and add more
if you like. For each site, start with the compass points close together and open them
gradually until you can reliably tell "two points" from "one", recording that gap in
millimetres.

In [ ]:
my_thresholds_mm = {
    "fingertip": 3,   # example data: replace with your own measured threshold, in mm
    "palm": 9,        # example data
    "forearm": 35,    # example data
    "back": 45,       # example data
}
mine = pd.Series(my_thresholds_mm, name="your threshold (mm)")
display(mine.to_frame())
mine.plot.barh(color="steelblue", figsize=(6, 2.4))
plt.xlabel("two-point discrimination threshold (mm)"); plt.gca().invert_yaxis(); plt.show()

## Compare

Figure 6.2 gives approximate published values (after Weinstein, 1968) for the same body parts:
2 mm at the fingertip, 10 mm on the palm, 30 mm on the forearm, and 40 mm on the upper back.

In [ ]:
book_values_mm = {"fingertip": 2, "palm": 10, "forearm": 30, "back": 40}   # Figure 6.2, after Weinstein (1968)
compare_df = pd.DataFrame({
    "your threshold (mm)": pd.Series(my_thresholds_mm),
    "book figure 6.2 (mm)": pd.Series(book_values_mm),
}).loc[["fingertip", "palm", "forearm", "back"]]
display(compare_df)
compare_df.plot.barh(figsize=(7, 3))
plt.xlabel("two-point discrimination threshold (mm)"); plt.gca().invert_yaxis(); plt.show()

ratio = (compare_df["your threshold (mm)"] / compare_df["book figure 6.2 (mm)"]).round(2)
print("Your threshold as a multiple of the book's approximate value at each site:")
print(ratio.to_string())
print("Everyone's absolute numbers vary with the tester's technique and your own sensitivity, so look at")
print("the pattern rather than the exact millimetres: fingertip should be much finer than back, by roughly")
print("the same order of magnitude in your data as in Figure 6.2.")

## The AI side

The book compares the body's uneven sensor density to event cameras, which report only the
pixels where brightness changes, rather than capturing full frames at a fixed rate. Below, a
simulated moving square is captured two ways: as full dense frames (like a conventional
camera, or your low-resolution back) and as sparse "events" only where something changes (like
an event camera, or your high-resolution fingertip).

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)
n_frames, size = 40, 64
positions = np.linspace(5, size - 5, n_frames).astype(int)

dense_frames = np.zeros((n_frames, size, size))
for i, p in enumerate(positions):
    dense_frames[i, p - 1:p + 2, p - 1:p + 2] = 1.0   # a small moving square

# Event-camera-style representation: only pixels whose value changed since the last frame.
events = np.abs(np.diff(dense_frames, axis=0))
dense_data_points = dense_frames.size
event_data_points = int((events > 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(dense_frames.sum(axis=0), cmap="Greys")
axes[0].set_title(f"Dense frames: {dense_data_points:,} values stored")
axes[1].imshow(events.sum(axis=0), cmap="Greys")
axes[1].set_title(f"Event-style: {event_data_points:,} non-zero values stored")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"Storing every frame in full needs about {dense_data_points / max(event_data_points, 1):.0f}x more"
      " numbers than storing only the pixels that changed.")

The event-style representation captures the same moving square with far fewer stored numbers,
because it only "reports" where something changed -- the same strategy your fingertip's dense
receptor field and your back's sparse one embody at opposite ends of the body. The analogy has
limits: a real event camera reports changes at microsecond resolution using specialised
hardware at every pixel, while this toy version just subtracts already-rendered frames after
the fact, and skin receptors respond to pressure and vibration, not to brightness change.

## Questions to think about

1. Your ratio column in the compare cell shows how many times coarser your back's threshold is than your fingertip's. Does that ratio roughly match Figure 6.2's ratio (40 mm / 2 mm = 20x), even if your absolute numbers differ?
2. The book says a violinist's cortical map of the fingering hand is larger than a non-musician's. If you repeated this test on your dominant versus non-dominant index finger, would you expect a measurable difference, and why might it be smaller than the difference between fingertip and back?
3. In the event-camera demo, what happens to the compression ratio (dense values divided by event values) if the square moves faster, so more pixels change between frames? What does that suggest about when an event-camera-like strategy saves the most, and when it saves little?
4. The homunculus distorts body size according to cortical representation rather than physical size. If you drew your own "homunculus" using your four thresholds as a guide (smaller threshold = bigger drawn body part), how would it compare to the classic image the text describes?

## Challenge

Test two-point discrimination on a body part not listed in Figure 6.2 -- the back of your hand,
your lips, or your calf -- and predict where it should fall relative to your other measurements
before you test it, based on how much fine motor or sensory work that body part normally does.

In [ ]:
# Example data: replace with your own prediction and measured threshold for a new body part.
new_site = "lips"
predicted_mm = 4   # example: your guess before testing
measured_mm = 5    # example data: your compass measurement, in mm

print(f"{new_site}: predicted {predicted_mm} mm, measured {measured_mm} mm")
combined = pd.concat([mine, pd.Series({new_site: measured_mm}, name="your threshold (mm)")])
combined.sort_values().plot.barh(color="steelblue", figsize=(6, 3))
plt.xlabel("two-point discrimination threshold (mm)"); plt.show()
print("How close was your prediction? A good prediction suggests you already had an intuitive sense of the")
print("body's uneven cortical map, even without having measured it directly before.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-51.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")